<!-- your logo here! -->
<img src="https://github.com/bionadmin/distrib/raw/main/core.png"/>

# Build dimwrpart Gold Table

##### Usage:
Note: You have "Include Table Create" disabled,
So be sure to run the schema update before running this notebook.

This file may have been auto generated.<br>
This template expects source data to exist in one or more Silver tables which<br>
the query in step "Get Source Data" will reference. The Silver tables should<br>
also be Delta Lake tables.<br><br>
Modify the query in step "Get Source Data" using the provided template as guidance.<br>
Once written, queries may be stored in a BION model or in a file directory with <br>
the name dimwrpart.sql so that subsequent builds do not overwrite the query<br>
with the template.<br><br>
The rest of the notebook is auto generated and may remain unchanged.<br>

#### Custom Beginning Cell

In [1]:
#Add custom python code into your Bion Table setting "Custom Beginning Cell" and it will appear here,
#Place %sql on the very first line to add custom sql code instead.
#Use <sourcedatabase>, <sourcetable>, <destinationdatabase>, <destinationtable> or <sourceschema> tokens to replace your code with the corresponding values
print("Custom Cell");

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 3, Finished, Available, Finished, False)

Custom Cell


#### Debug and Dev Options

In [2]:
#Be sure to set debugMode to False when not debugging -- especially if you recieve "The output of the notebook is too large" errors.
#Definitely set to false in production.
debug_mode = False # Include debug select and print statements

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 4, Finished, Available, Finished, False)

####  Define Session and Table-specific Variables

In [3]:
import uuid
from datetime import datetime

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 5, Finished, Available, Finished, False)

In [4]:
df= spark.sql("select * from LH_BI_GOLD.dimwrpart ")
if df.count() != 0:
    spark.sql("delete from LH_BI_GOLD.dimwrpart")

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 6, Finished, Available, Finished, False)

In [5]:
#Notebook Parameters
run_grouping_id=""
run_dttm=""

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 7, Finished, Available, Finished, False)

In [6]:
#Session Variables

destination_database_name="LH_BI_GOLD"
destination_table_name="dimwrpart"
# Capture the RunID So that table logs can be grouped by run
if not run_grouping_id:
    if debug_mode:
        print("Grouping ID from parent not found. Creating new Grouping ID")
    run_grouping_id = str(uuid.uuid4())
else:
    if debug_mode:
        print("Parent Grouping ID found.")

if not run_dttm:
    if debug_mode:
        print("Start DTTM from parent not found. Creating new Start DTTM")
    run_dttm = str(datetime.now())
else:
    if debug_mode:
        print("Parent Grouping ID found.")

if debug_mode:
    print("Grouping ID:"+run_grouping_id)    
spark.conf.set("sess.NB_GOLD_dimwrpart_run_grouping_id",run_grouping_id)
spark.conf.set("sess.NB_GOLD_dimwrpart_run_dttm",run_dttm)

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 8, Finished, Available, Finished, False)

#### Misc. Spark Options
If generating code, edit these in the template settings for consistency<br>
across all notebooks.

In [7]:
spark.conf.set("spark.sql.parquet.vorder.enabled","false")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled","true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize","1073741824")

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 9, Finished, Available, Finished, False)

In [8]:
%%sql
SET ANSI_MODE = TRUE

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

#### Log table processing start

In [9]:
%%sql
WITH SRC AS (
  SELECT 
    '${sess.NB_GOLD_dimwrpart_run_grouping_id}' AS GroupingID,
    '${sess.NB_GOLD_dimwrpart_run_dttm}' AS StartDateTime,
    'LH_BI_GOLD' AS DatabaseName,
    'dimwrpart' AS TableName,
    current_timestamp() AS TableStartTime,
    (select count(*) from LH_BI_GOLD.dimwrpart) AS RowsBefore
    WHERE NOT EXISTS (
        SELECT GroupingID from LH_BI_GOLD.Audit_RuntimeTableLog WHERE GroupingID = '${sess.NB_GOLD_dimwrpart_run_grouping_id}' 
        AND DatabaseName = 'LH_BI_GOLD' 
        AND TableName = 'dimwrpart' 
    )
    
) 
INSERT INTO LH_BI_GOLD.Audit_RuntimeTableLog
(
  GroupingID,
  StartDateTime,
  DatabaseName,
  TableName,
  TableStartTime,
  TableEndTime,
  TableDurationSeconds,
  RowsAffected,
  RowsInserted,
  RowsUpdated,
  RowsDeleted,
  RowsBefore,
  RowsAfter,
  Audit_Status,
  Audit_CreatedDateTime,
  Audit_ModifiedDateTime
)
SELECT 
  SRC.GroupingID,
  SRC.StartDateTime,
  SRC.DatabaseName,
  SRC.TableName,
  SRC.TableStartTime,
  SRC.TableStartTime,
  0,
  0,
  0,
  0,
  0,
  SRC.RowsBefore,
  0,
  'Processing',
  SRC.TableStartTime,
  SRC.TableStartTime
FROM SRC;

--In cases where we are restarting a specific GroupingID run...
UPDATE LH_BI_GOLD.Audit_RuntimeTableLog
  SET Audit_Status = 'Processing'
WHERE GroupingID = '${sess.NB_GOLD_dimwrpart_run_grouping_id}'
  AND DatabaseName = 'LH_BI_GOLD' 
  AND TableName = 'dimwrpart' 
  AND Audit_Status <> 'Processing'

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 12, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 1 fields>

#### Get High water mark based on audit fields.
<BR>Modify query as needed.

In [10]:
max_audit_time_df=spark.sql("select COALESCE(MAX(Audit_Source_ModifiedDateTime),CAST('1901-01-01' AS TIMESTAMP)) as max_audit_time FROM "+destination_database_name+"."+destination_table_name)
#display(max_audit_time_df)
max_audit_time = max_audit_time_df.first()['max_audit_time']
spark.conf.set("sess.NB_GOLD_dimwrpart_max_audit_time",str(max_audit_time))

if debug_mode:
    print (max_audit_time)

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 13, Finished, Available, Finished, False)

#### Custom Pre Query Cell

In [11]:
#Add custom python code into your Bion Table setting "Custom Pre Query Cell" and it will appear here,
#Place %sql on the very first line to add custom sql code instead.
#Use <sourcedatabase>, <sourcetable>, <destinationdatabase>, <destinationtable> or <sourceschema> tokens to replace your code with the corresponding values
print("Custom Cell");

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 14, Finished, Available, Finished, False)

Custom Cell


#### Get Source Data
<BR>Modify query as needed.

In [12]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW src_data_for_dimwrpart AS
SELECT   9999 as PCN,
         a.Projectid as Part_Key,
         b.projectidentifier as Part_No,
         pm.captiondefault as name,
         null as revision,
         pm.Prod_family as Part_type,
         b.WorkflowResolved as Part_status,
         null as unit,
         null as building_key,
         pm.Prod_group as Part_group_key, -- to have same value for all rows as corresponding to inprocess status
         pm.Prod_group as Part_Group,
         pm.Program as Product_type_key,
         pm.Program as Product_type,
         null as Product_type_Code,
         null as Part_source_key,
         null as Purchased_Part,
         null as Manufactured_Part,
         'FACTON' as Source,
        GREATEST(a.Audit_ModifiedDatetime, 
              b.Audit_ModifiedDatetime,
              COALESCE(pm.Audit_ModifiedDatetime, a.Audit_ModifiedDatetime)) AS Audit_ModifiedDatetime
      ,current_timestamp() AS RIGHTNOW 
FROM LH_BI_SILVER_TIER1.wrproject a
LEFT JOIN LH_BI_SILVER_TIER1.wrversioninfo b 
    ON a.Projectid = b.ProjectVersionID 
   -- AND LOWER(b.WorkflowResolved) in ('par','development','readiness','post launch','launch','award','in process') removing this filter bcs europe is using other gates as well
LEFT JOIN (
    SELECT 
        bb.projectidentifier,  
        aa.captiondefault,
        aa.Prod_family,
        aa.Prod_group,
        aa.Program,
        aa.Audit_ModifiedDatetime
    FROM LH_BI_SILVER_TIER1.wrproject aa 
    JOIN LH_BI_SILVER_TIER1.wrversioninfo bb 
        ON aa.Projectid = bb.ProjectVersionID 
    WHERE (lower(bb.WorkflowResolved) = 'in process')
        AND bb.NextProjectVersionID is null
) pm ON b.projectidentifier = pm.projectidentifier -- Join on projectidentifier

/*where
(
        a.Audit_ModifiedDateTime>CAST('${sess.NB_GOLD_dimwrpart_max_audit_time}' AS TIMESTAMP)
        or b.Audit_ModifiedDateTime>CAST('${sess.NB_GOLD_dimwrpart_max_audit_time}' AS TIMESTAMP ) )
        --Global Delivery
      
      --TEMPLATE BELOW:
SELECT 
          --*******************************************************
          --BEGIN CUSTOM BLOCK: Edit the following query as needed.
          --Be sure to keep the column aliases which should match the
          --destination column name(s)

          --If generating this notebook, this query can be stored in 
          --the data model in a table setting named "Gold Table Query".
          --This will allow you to persist this query while
          --generating these notebooks. Note that if columns are added or removed from
          --the main table, the queries must be modified.
          --*******************************************************
          t1.c0 AS PartNumber,
          t1.c1 AS PartSite,
          t1.c2 AS PlexPartKey,
          t1.c3 AS PlexPCN,
          t1.c4 AS Unit,
          t1.c5 AS Name,
          t1.c6 AS SilverPartKey,
        Audit_ModifiedDateTime,   
          current_timestamp() AS RIGHTNOW--leave this.
      FROM Silver_sales.Some_Silver_Table t1
      --LEFT OUTER JOIN Silver_sales.Some_OTHER_Silver_Table T2
      --ON T1.SomeValue = T2.SomeValue
      WHERE (
        t1.Audit_ModifiedDateTime>CAST('${sess.NB_GOLD_dimwrpart_max_audit_time}' AS TIMESTAMP)
      )
      -- AND SomethingIsPerhaps > Something Or SomethingElse
      --*******************************************************
      --END CUSTOM BLOCK
      --*******************************************************

      */

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

#### Review the Source Data
<BR>Modify query as desired to review the source data.

#### Upsert Data

In [13]:
_bionSqlResults = spark.sql( """

MERGE INTO LH_BI_GOLD.dimwrpart DST
USING (
  SELECT
   *
   FROM (
     SELECT unhex(wrpartKey__HEXSTRING) AS wrpartKey,
           cast(conv(left(wrpartKey__HEXSTRING,15),16,10) as bigint) AS wrpartKeyInt,
           PCN, Part_Key, Part_No, name, revision, Part_type, Part_status, unit, building_key, Part_group_key, Part_Group, Product_type_key, Product_type, Product_type_Code, Part_source_key, Purchased_Part, Manufactured_Part, Source,
           Audit_ModifiedDateTime as Audit_Source_ModifiedDateTime,
           ROW_NUMBER() OVER (PARTITION BY Part_key, Part_No ORDER BY Audit_Source_ModifiedDateTime DESC) DUPLICATES_RN,
           RIGHTNOW  
     FROM (
       SELECT
          SHA2(
              CAST(`PCN` AS STRING)|| '|'||
              CAST(`Part_No` AS STRING)
             ,256) AS wrpartKey__HEXSTRING,
           CAST(PCN AS INT) AS PCN, CAST(Part_Key AS INT) AS Part_Key, CAST(COALESCE(Part_No, '') AS STRING) AS Part_No, CAST(COALESCE(name, '') AS STRING) AS name, 
           CAST(COALESCE(revision, '') AS STRING) AS revision, CAST(COALESCE(Part_type, '') AS STRING) AS Part_type, CAST(COALESCE(Part_status, '') AS STRING) AS Part_status, 
           CAST(COALESCE(unit, '') AS STRING) AS unit, CAST(building_key AS INT) AS building_key, CAST(COALESCE(Part_group_key, '') AS STRING) AS Part_group_key, 
           CAST(COALESCE(Part_Group, '') AS STRING) AS Part_Group, CAST(COALESCE(Product_type_key, '') AS STRING) AS Product_type_key, 
           CAST(COALESCE(Product_type, '') AS STRING) AS Product_type, CAST(COALESCE(Product_type_Code, '') AS STRING) AS Product_type_Code, 
           CAST(COALESCE(Part_source_key, '') AS STRING) AS Part_source_key, CAST(Purchased_Part AS BOOLEAN) AS Purchased_Part, 
           CAST(Manufactured_Part AS BOOLEAN) AS Manufactured_Part, CAST(Source AS STRING) AS Source, CAST(Audit_ModifiedDateTime AS TIMESTAMP) AS Audit_ModifiedDateTime,
           CAST(Audit_ModifiedDateTime as TIMESTAMP) AS Audit_Source_ModifiedDateTime, current_timestamp() AS RIGHTNOW
       FROM src_data_for_dimwrpart
     ) SRC1
   ) SRC2 
   WHERE DUPLICATES_RN = 1
 ) SRC
ON SRC.Part_key = DST.Part_key
and SRC.Part_No = DST.Part_No

WHEN NOT MATCHED THEN INSERT (
    wrpartKey,wrpartKeyInt,PCN, Part_Key, Part_No, name, revision, Part_type, Part_status, unit, building_key, Part_group_key, Part_Group, 
    Product_type_key, Product_type, Product_type_Code, Part_source_key, Purchased_Part, Manufactured_Part, Source, Audit_ModifiedDateTime, Audit_Source_ModifiedDateTime
)
VALUES(
    SRC.wrpartKey,SRC.wrpartKeyInt,SRC.PCN, SRC.Part_Key, SRC.Part_No, SRC.name, SRC.revision, SRC.Part_type, SRC.Part_status, 
    SRC.unit, SRC.building_key, SRC.Part_group_key, SRC.Part_Group, SRC.Product_type_key, SRC.Product_type,
    SRC.Product_type_Code, SRC.Part_source_key, SRC.Purchased_Part, SRC.Manufactured_Part, SRC.Source, SRC.RIGHTNOW, SRC.Audit_Source_ModifiedDateTime
)
""")

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 16, Finished, Available, Finished, False)

In [14]:
rowsupserteddf=None
try:
    rowsupserteddf = _bionSqlResults
except:
    #catch the not defined error
    print("Warning: _bionSqlResults not defined")
    rowsupserteddf = spark.sql("select CAST(-1 as BIGINT) as num_affected_rows, CAST(-1 as BIGINT) as num_updated_rows, CAST(-1 as BIGINT) as num_deleted_rows, CAST(-1 as BIGINT) as num_inserted_rows") 

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 17, Finished, Available, Finished, False)

#### Log table completion

In [15]:
rowsupserteddf.createOrReplaceTempView("tmp_LH_BI_GOLD_dimwrpart_upserted");
if ("rowsdeleteddf" in locals()):
    rowsdeleteddf.createOrReplaceTempView("tmp_LH_BI_GOLD_dimwrpart_deleted")
else:
    #cannot make another view off of rowsupserteddf as this will fail when attempting to join two views from the same dataframe. Thus:
    spark.sql("CREATE OR REPLACE TEMPORARY VIEW tmp_LH_BI_GOLD_dimwrpart_deleted AS (SELECT * FROM tmp_LH_BI_GOLD_dimwrpart_upserted)")
    

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 18, Finished, Available, Finished, False)

In [16]:
%%sql

UPDATE LH_BI_GOLD.Audit_RuntimeTableLog 
SET
  TableEndTime = current_timestamp(),
  TableDurationSeconds = datediff(SECOND,TableStartTime, current_timestamp()),
  RowsAffected = COALESCE(RowsAffected,0)+(
    select u.num_affected_rows+d.num_deleted_rows 
    FROM tmp_LH_BI_GOLD_dimwrpart_upserted u
    CROSS JOIN tmp_LH_BI_GOLD_dimwrpart_deleted d
  ),
  RowsInserted = COALESCE(RowsInserted,0)+(select num_inserted_rows from tmp_LH_BI_GOLD_dimwrpart_upserted),
  RowsUpdated = COALESCE(RowsUpdated,0)+(select num_updated_rows from tmp_LH_BI_GOLD_dimwrpart_upserted),
  RowsDeleted = COALESCE(RowsDeleted,0)+(select num_deleted_rows from tmp_LH_BI_GOLD_dimwrpart_deleted),
  RowsAfter = (select count(*) from LH_BI_GOLD.dimwrpart),
  Audit_Status = 'Completed',
  Audit_ModifiedDateTime = current_timestamp()
WHERE
  GroupingID = '${sess.NB_GOLD_dimwrpart_run_grouping_id}'
  AND DatabaseName = 'LH_BI_GOLD'
  AND TableName = 'dimwrpart'

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 19, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

#### Review the Upserted Data
<BR>Modify query as desired to review the upserted data.

In [17]:
if debug_mode:
    display(spark.sql("select * from LH_BI_GOLD.dimwrpart ORDER BY Audit_ModifiedDateTime DESC LIMIT 1000"))

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 20, Finished, Available, Finished, False)

#### Custom End Cell

In [18]:
#Add custom python code into your Bion Table setting "Custom End Cell" and it will appear here,
#Place %sql on the very first line to add custom sql code instead.
#Use <sourcedatabase>, <sourcetable>, <destinationdatabase>, <destinationtable> or <sourceschema> tokens to replace your code with the corresponding values
print("Custom Cell");

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 21, Finished, Available, Finished, False)

Custom Cell


In [19]:
notebookutils.notebook.exit("Complete");

StatementMeta(, 198d68ab-82c5-4b94-a72b-972ba8d328f2, 22, Finished, Available, Finished, False)

ExitValue: Complete

#### End of Notebook